# Somo la 18 (linalofuata): Risiti Zinazoonyesha *Mtu* Aliyekubali Kitendo

Somo linaonyesha nini **wakala** alifanya na nini **mlango** uliamua. Daftari hili linaongeza nusu iliyokosekana: ushahidi kwamba **mtu mwenye jina** alikubali kitendo **sahihi** — saini tofauti inayoshikiliwa na mtu juu ya kitendo kamili cha kanoni, iliyohakikiwa kwa njia isiyo mtandao.

Vitu vyote viwili hapa vinatumia **umbo moja la mfuko kama risiti za somo**: mzigo ulio wima na uwanja la `type`, saini ya Ed25519 juu ya bayti wa JCS wa kanoni moja kwa moja, pamoja na kitu cha `signature` kilicho na muundo kilichowekwa (na kimeondolewa kutoka kwa bayti zilizosainiwa). Risiti ya idhini ni `type` mpya (`human.approval.v1`) sambamba na aina ya kitendo, hivyo `verify_chain` moja inashughulikia aina zote mbili za vitu kwa njia sawa ya msimbo uliyojenga kwenye daftari kuu. Risiti hii ya idhini ya mtu ni muundo wa elimu ulioainishwa hapa, si aina ya risiti iliyoainishwa na draft-farley-acta-signed-receipts.

Maboresho moja makusudi juu ya mtathmini wa maonyesho katika daftari kuu: mtathmini hapa humtumia `signature.key_id` dhidi ya **rejista ya funguo zilizowekwa** badala ya kuamini funguo ya umma iliyobebwa ndani ya risiti. Hilo ndilo mtazamo wa utengenezaji ambao orodha ya kuangalia ya somo inapendekeza ("chapisha funguo ya umma ya uthibitishaji"), na hilo ndilo linalofanya uongozi wa udanganyifu kuwa kukataa badala ya njia ya kuingiza funguo zako mwenyewe.

Kanuni ambayo daftari hili linalifundisha: **idhini iliyosainiwa si mamlaka yenyewe.** Mamlaka ipo tu ikiwa risiti ya idhini na risiti ya kitendo bado zinahusiana na kitendo hicho kamili cha kanoni wakati wa utekelezaji, chini ya toleo la sera, ufunguo, na muda wa kumalizika ambao bado ni wa sasa, na idhini ambayo bado haijatumiwa. Kila kushindwa kuruhusu kwa sababu **tofauti**, hivyo unaweza kuelewa *mamlaka imepitwa na wakati* tofauti na *kitendo kilichotekelezwa kimebadilika*.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## Hatua halisi

Kitengo cha idhini ni **kitu cha kitendo halisi** — si lebo isiyoeleweka kama "idhinisha marejesho," bali kitendo kamili, kilichobainishwa kikamilifu. Kusaini kitu kizima (na kutoa muhtasari kutoka kwacho) ndiyo inatuwezesha baadaye kuthibitisha kuwa mwanadamu alikubali *hiki* na si chochote kingine.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## Mfuko mmoja, mamlaka mbili

Kila risiti ni mfuko wa somo: mzigo usio na mwinuko wenye uwanja wa `type`, pamoja na kitu cha `signature` (`alg`, `sig`, `key_id`) ambacho si sehemu ya baiti zilizosainiwa. `verify_envelope` ni ukaguzi wa muundo na saini ya pamoja kwa aina zote mbili za risiti; ni **rejista ya funguo zilizowekwa** ambayo inatambua `signature.key_id` ndio huweka mamlaka tofauti:

- **risiti ya idhini** (`human.approval.v1`) — mtoaji aliyebainishwa, kitendo kamili cha kanoni **na muhtasari wake**, `policy_version`, tarehe ya kutolewa + kumalizika kwa muda. Matumizi ya mara moja yanasimbuliwa ngazi ya mlolongo.
- **risiti ya kitendo** (`agent.action.v1`) — utambulisho wa wakala, `run_id`, muhtasari huo huo wa kitendo cha kanoni, matokeo ya utekelezaji + muda, na `parent_approval_ref`: `receipt_hash` ya idhini, desturi ile ile kama `previous_receipt_hash` katika mlolongo wa somo.

Uwanja wa pamoja wa `action_digest` ndio mshikamano ambao mteketezi hutegemea. `key_id` huishi katika kitu cha saini kama kidokezo cha kutafuta tu: kuielekeza upya kwenye funguo tofauti iliyowekwa hufanya ukaguzi wa saini kushindwa, kwa hivyo haileti kitu.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: mahali ambapo uhusiano huamuliwa kabisa

`verify_chain` si **kifukizio cha urahisi** cha ukaguzi wa saini mbili. Ni sehemu moja ambapo `action_digest` rasmi inayoshirikiwa, sera/ufunguo/ukinifu wa **ubora** wa idhini, na **matumizi ya mara moja** ya idhini hupimwa pamoja, dhidi ya tendo linalotekelezwa *sasa hivi*.

Kila kushindwa hupinga kwa **sababu tofauti**, hivyo msomaji wa kukataa anaweza kusema ikiwa mamlaka imepitwa na wakati (sera imesogezwa, ufunguo umepinduliwa, idhini imepitwa na wakati, idhini imetumiwa) au tendo lililotekelezwa limebadilika wakati idhini bado ni halali (badiliko la digest).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## Kile kifungo kinachokamata

Kila kesi hapa chini inashindwa **kufunguliwa** kwa **sababu tofauti**. Kipande cha kwanza ni seti ya kawaida (kuingilia, mdhamini aliyechanganyikiwa, kurudisha ujumbe, ulaghai kwa mamlaka yoyote, data isiyo sahihi). Kipande cha pili ni jozi inayofanya mali hiyo kuwa halisi badala ya kudaiwa:

- **mamlaka iliyochoka** — saini bado ni halali, lakini toleo la sera limebadilika, ufunguo wa mtoaji ruhusa umeondolewa kwenye rejista iliyowekwa, au ruhusa ilikuja kumalizika kabla ya utekelezaji;
- **badilisho la sarufi** — risiti ya kitendo iliyosainiwa kwa usahihi ambayo `parent_approval_ref` inaelekeza kwenye ruhusa *halisi*, lakini sarufi ya kitendo ya ruhusa hiyo siyo ile ile yenye kitendo kinachotekelezwa.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## Hii inathibitisha nini — na haithibitishi nini

**Inathibitisha:** binadamu aliyepewa jina alikubali *hatua hii halali kabisa* (hatua kamili + muhtasari, iliyosainiwa na ufunguo ulioamuliwa kutoka kwa rejista iliyoshikiliwa), na wakala alitekeleza *hasa hatua hiyo iliyoruhusiwa* (muhtasari uleule, risiti iliyounganishwa na idhini kwa `receipt_hash`, kanuni ya mnyororo wa somo) — wakati toleo la sera la idhini, ufunguo, na muda wake wa kumalizika vilikuwa bado vya sasa, mara moja tu. Ikiwa upande wowote utakabadilika, mnyororo unafungwa, na sababu ya kukataa inakuambia **ipi** mali iliyovunjika: mamlaka ya zamani dhidi ya hatua iliyobadilika.

**Haithibitishi:** kwamba UI ya idhini ilimuonyesha binadamu kile walichodhani wanakisaini (WYSIWYS ni tatizo la pekee), kwamba ufunguo haukukutwa kwa nguvu au kuibiwa kabla ya mzunguko, au kwamba athari za chini zililingana na hatua. Kusainiwa ≠ kuidhinishwa: sahihi ya saini juu ya sera iliyozidiwa, ufunguo uliobadilishwa, dirisha lililotoweka muda wake, au muhtasari tofauti haina maana hapa.

Aina mbili za risiti zinashiriki kifuko cha somo na njia moja ya msimbo wa `verify_chain` kwa makusudi: uunganisho ulioumba kwa risiti za hatua katika daftari kuu ni msimbo ule ule unaokagua idhini ya binadamu. Mkataba mmoja wa ukaguzi, mamlaka tofauti zilizoshikiliwa, zikunganishwa na muhtasari wa hatua halali na hakuna kitu kingine.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
